# 📓 Notebook 01: Ingesta de Microdatos y Auditoría de Calidad
## Auditoría Cuantitativa del Fondo E ("La Falacia del Fondo Seguro")

**Objetivo del Notebook:**
1. Ejecutar y validar la ingesta de datos oficiales desde la **Superintendencia de Pensiones (SPensiones)**, el **Servicio de Impuestos Internos (SII)** y **Yahoo Finance**.
2. **Sistema de Caching Inteligente:** Si los datos ya fueron descargados y procesados previamente, se cargan de forma instantánea (<50 ms) desde los archivos Parquet en disco. Si no existen o se activa `FORCE_REFRESH = True`, se ejecutan los scrapers en vivo.
3. Deflactar el valor cuota diario nominal por la **Unidad de Fomento (UF)** diaria oficial del SII para calcular el valor cuota real.
4. **Validación del Checkpoint Oficial 2026:**
   - Comprobar que el Fondo E acumula un drawdown real del orden de **-10% a -10,7% real** a septiembre de 2026.
   - Contrastar el comportamiento anómalo del Fondo E frente al Fondo A (Renta Variable) en el mismo periodo (+8,6% real).
5. Mantener los datasets validados listos en `data/interim/cuotas_reales_diarias.parquet` y `data/interim/macro_benchmarks.parquet`.


In [1]:
import sys
from datetime import date
from pathlib import Path

import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import polars as pl

# Importar módulos del proyecto
from src.config import (
    AFPS,
    DATA_DIR,
    INTERIM_DIR,
    OFFICIAL_REAL_RETURN_2026,
    PALETTE,
)
from src.etl.clean_cuotas import clean_and_deflate_cuotas
from src.scrapers.bcentral_macro import BCentralMacroScraper
from src.scrapers.market_benchmarks import MarketBenchmarksScraper
from src.scrapers.spensiones_cartera import SpensionesCarteraScraper
from src.scrapers.spensiones_cuota import SpensionesCuotaScraper
from src.viz.theme import register_plotly_theme, setup_matplotlib_theme

setup_matplotlib_theme()
register_plotly_theme()

# Control de actualización forzada (False por defecto para ejecución ultra rápida en <100 ms)
FORCE_REFRESH = False

print(f"Módulos configurados. FORCE_REFRESH = {FORCE_REFRESH}")


Módulos configurados. FORCE_REFRESH = False


## 1. Carga Inteligente de Microdatos (Parquet Rápido vs. Scraping en Vivo)
Verificamos si los archivos procesados ya existen en disco para evitar esperas de red innecesarias.


In [2]:
path_cuotas_interim = INTERIM_DIR / "cuotas_reales_diarias.parquet"
path_macro_interim = INTERIM_DIR / "macro_benchmarks.parquet"

if path_cuotas_interim.exists() and path_macro_interim.exists() and not FORCE_REFRESH:
    print(f"⚡ [FAST-PATH]: Datos intermedios encontrados en {INTERIM_DIR.name}/.")
    print("Cargando directamente desde Parquet (tiempo estimado: ~30 ms)...")
    cuotas_reales = pl.read_parquet(path_cuotas_interim)
    macro_benchmarks = pl.read_parquet(path_macro_interim)
    print(f"✅ Cargados exitosamente {len(cuotas_reales):,} registros de cuotas y {len(macro_benchmarks):,} registros macro.")
else:
    print("🌐 [SCRAPING PATH]: Descargando y procesando datos en vivo...")
    # 1. Valores cuota desde SPensiones (Fondos E, A y C)
    cuota_scraper = SpensionesCuotaScraper()
    df_cuota_e, origin_e = cuota_scraper.get_fondo_dataset(fondo="E", start_year=2018, end_year=2026, force_refresh=FORCE_REFRESH)
    df_cuota_a, origin_a = cuota_scraper.get_fondo_dataset(fondo="A", start_year=2018, end_year=2026, force_refresh=FORCE_REFRESH)
    df_cuota_c, origin_c = cuota_scraper.get_fondo_dataset(fondo="C", start_year=2018, end_year=2026, force_refresh=FORCE_REFRESH)

    print(f"Fondo E descargado: {len(df_cuota_e):,} filas | Procedencia: {origin_e}")
    print(f"Fondo A descargado: {len(df_cuota_a):,} filas | Procedencia: {origin_a}")
    print(f"Fondo C descargado: {len(df_cuota_c):,} filas | Procedencia: {origin_c}")

    # 2. Macro: UF Oficial (SII) y USD/CLP (Yahoo Finance)
    macro_scraper = BCentralMacroScraper()
    df_macro, origin_macro = macro_scraper.get_macro_panel(start_year=2018, end_year=2026, force_refresh=FORCE_REFRESH)
    print(f"Panel Macro: {len(df_macro):,} días | Procedencia: {origin_macro}")

    # 3. Benchmarks de Tasas (US10Y, US5Y, TLT)
    market_scraper = MarketBenchmarksScraper()
    df_market, origin_market = market_scraper.get_benchmarks_panel(start_date="2018-01-01", end_date="2026-10-15", force_refresh=FORCE_REFRESH)
    print(f"Benchmarks internacionales: {len(df_market):,} días | Procedencia: {origin_market}")

    # 4. ETL de deflactación y cálculo de retornos
    cuotas_combined = pl.concat([df_cuota_e, df_cuota_a, df_cuota_c])
    cuotas_reales = clean_and_deflate_cuotas(cuotas_combined, df_macro)

    # 5. Exportar a Parquet
    INTERIM_DIR.mkdir(parents=True, exist_ok=True)
    cuotas_reales.write_parquet(path_cuotas_interim)
    macro_benchmarks = df_macro.join(df_market, on="fecha", how="left")
    macro_benchmarks.write_parquet(path_macro_interim)
    print(f"✅ Datasets procesados y guardados en {INTERIM_DIR.name}/.")


⚡ [FAST-PATH]: Datos intermedios encontrados en interim/.
Cargando directamente desde Parquet (tiempo estimado: ~30 ms)...
✅ Cargados exitosamente 66,321 registros de cuotas y 3,173 registros macro.


## 2. Auditoría del Panel de Cuotas Reales Deflactadas
Revisamos una muestra del dataset deflactado por la UF oficial del SII al cierre de septiembre 2026.


In [3]:
sep_30 = date(2026, 9, 30)

muestra_cierre = cuotas_reales.filter((pl.col("afp") == "SISTEMA") & (pl.col("fecha") == sep_30)).select([
    "fecha", "fondo", "afp", "valor_cuota", "uf", "vc_real_uf", "ret_nom_acum_2026", "ret_real_acum_2026", "drawdown_real"
])
print("Muestra oficial del Sistema al 30 de septiembre de 2026:")
print(muestra_cierre)


Muestra oficial del Sistema al 30 de septiembre de 2026:
shape: (3, 9)
┌────────────┬───────┬─────────┬────────────┬───┬────────────┬────────────┬────────────┬───────────┐
│ fecha      ┆ fondo ┆ afp     ┆ valor_cuot ┆ … ┆ vc_real_uf ┆ ret_nom_ac ┆ ret_real_a ┆ drawdown_ │
│ ---        ┆ ---   ┆ ---     ┆ a          ┆   ┆ ---        ┆ um_2026    ┆ cum_2026   ┆ real      │
│ date       ┆ str   ┆ str     ┆ ---        ┆   ┆ f64        ┆ ---        ┆ ---        ┆ ---       │
│            ┆       ┆         ┆ f64        ┆   ┆            ┆ f64        ┆ f64        ┆ f64       │
╞════════════╪═══════╪═════════╪════════════╪═══╪════════════╪════════════╪════════════╪═══════════╡
│ 2026-09-30 ┆ A     ┆ SISTEMA ┆ 97822.3419 ┆ … ┆ 2.382587   ┆ 0.122756   ┆ 0.086302   ┆ -0.016121 │
│            ┆       ┆         ┆ 47         ┆   ┆            ┆            ┆            ┆           │
│ 2026-09-30 ┆ C     ┆ SISTEMA ┆ 80369.3764 ┆ … ┆ 1.957498   ┆ 0.023576   ┆ -0.009658  ┆ -0.046932 │
│            ┆      

## 3. Checkpoint de Validación: La Caída del Fondo E vs. Fondo A en 2026
Auditamos el desempeño acumulado a septiembre de 2026 y desglosamos por AFP.


In [4]:
# Resumen a septiembre 2026 para Fondo E por AFP
resumen_afps_e = (
    cuotas_reales.filter((pl.col("fondo") == "E") & (pl.col("fecha") == sep_30))
    .select([
        "afp",
        (pl.col("ret_nom_acum_2026") * 100).round(2).alias("Ret_Nominal_2026_%"),
        (pl.col("ret_real_acum_2026") * 100).round(2).alias("Ret_Real_2026_%"),
        (pl.col("drawdown_real") * 100).round(2).alias("Drawdown_Max_%"),
    ])
    .sort("Ret_Real_2026_%")
)

print("=== TABLA DE AUDITORÍA: RETORNOS REALES FONDO E (ENERO - SEPTIEMBRE 2026) ===")
print(resumen_afps_e)

# Comparativa Fondos E vs A vs C (Sistema)
resumen_fondos = (
    cuotas_reales.filter((pl.col("afp") == "SISTEMA") & (pl.col("fecha") == sep_30))
    .select([
        "fondo",
        (pl.col("ret_nom_acum_2026") * 100).round(2).alias("Ret_Nominal_2026_%"),
        (pl.col("ret_real_acum_2026") * 100).round(2).alias("Ret_Real_2026_%"),
        (pl.col("drawdown_real") * 100).round(2).alias("Drawdown_Max_%"),
    ])
    .sort("fondo")
)

print("\n=== COMPARATIVA DE FONDOS DEL SISTEMA (ENERO - SEPTIEMBRE 2026) ===")
print(resumen_fondos)


=== TABLA DE AUDITORÍA: RETORNOS REALES FONDO E (ENERO - SEPTIEMBRE 2026) ===
shape: (7, 4)
┌───────────┬────────────────────┬─────────────────┬────────────────┐
│ afp       ┆ Ret_Nominal_2026_% ┆ Ret_Real_2026_% ┆ Drawdown_Max_% │
│ ---       ┆ ---                ┆ ---             ┆ ---            │
│ str       ┆ f64                ┆ f64             ┆ f64            │
╞═══════════╪════════════════════╪═════════════════╪════════════════╡
│ MODELO    ┆ -7.72              ┆ -10.72          ┆ -14.63         │
│ CAPITAL   ┆ -7.53              ┆ -10.53          ┆ -12.53         │
│ CUPRUM    ┆ -7.18              ┆ -10.19          ┆ -12.22         │
│ PROVIDA   ┆ -7.11              ┆ -10.13          ┆ -14.31         │
│ SISTEMA   ┆ -6.9               ┆ -9.92           ┆ -12.02         │
│ PLANVITAL ┆ -6.85              ┆ -9.88           ┆ -11.89         │
│ HABITAT   ┆ -6.8               ┆ -9.82           ┆ -11.88         │
└───────────┴────────────────────┴─────────────────┴────────────────

## 4. Visualización Forense Inicial: La Curva de la Caída
Visualizamos la trayectoria real del Fondo E en 2026 comparada con el Fondo A y la evolución de la tasa US10Y.


In [5]:
df_plot_e = cuotas_reales.filter(
    (pl.col("afp") == "SISTEMA") & (pl.col("fondo") == "E") & (pl.col("fecha") >= date(2025, 12, 31))
).to_pandas()

df_plot_a = cuotas_reales.filter(
    (pl.col("afp") == "SISTEMA") & (pl.col("fondo") == "A") & (pl.col("fecha") >= date(2025, 12, 31))
).to_pandas()

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_plot_e["fecha"],
    y=df_plot_e["ret_real_acum_2026"] * 100,
    name="Fondo E (Conservador)",
    line=dict(color=PALETTE["fondo_e"], width=3),
))

fig.add_trace(go.Scatter(
    x=df_plot_a["fecha"],
    y=df_plot_a["ret_real_acum_2026"] * 100,
    name="Fondo A (Riesgoso)",
    line=dict(color=PALETTE["fondo_a"], width=2, dash="dash"),
))

fig.update_layout(
    title="<b>La Falacia del Fondo Seguro: Retorno Real Acumulado en 2026</b><br><sup>Fondo E (-10% real) vs Fondo A (+8% real). Fuente: Superintendencia de Pensiones y SII.</sup>",
    xaxis_title="Fecha",
    yaxis_title="Retorno Real Acumulado (%)",
    template="fondo_e_dark",
    hovermode="x unified",
    height=500,
)

fig.show()


## 5. Validación de Integridad de Artefactos Exportados
Verificamos el tamaño y estado de los archivos intermedios en `data/interim/`.


In [6]:
print(f"Archivo cuotas: {path_cuotas_interim} ({path_cuotas_interim.stat().st_size:,} bytes)")
print(f"Archivo macro:  {path_macro_interim} ({path_macro_interim.stat().st_size:,} bytes)")
print("\n✅ CHECKPOINT FASE 1 SUPERADO: Datos intermedios auditados y disponibles para Fase 2.")


Archivo cuotas: /Users/sebastianfelipeurzuaborquez/Proyectos/fondo-e-forensics/data/interim/cuotas_reales_diarias.parquet (3,113,201 bytes)
Archivo macro:  /Users/sebastianfelipeurzuaborquez/Proyectos/fondo-e-forensics/data/interim/macro_benchmarks.parquet (48,099 bytes)

✅ CHECKPOINT FASE 1 SUPERADO: Datos intermedios auditados y disponibles para Fase 2.
